# Fine-tune Qwen3.5-2B — x2f recipe, 2026-10-24

Same training setup as `x2f` (`finetune-colab-2026-10-13.ipynb`):
417 rows, LoRA r=16 / alpha=32 on 7 projections, lr 2e-4, responses-only,
epochs derived to hold ~1968 presentations, seed 3407.

Only the base model changes: `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` → `unsloth/Qwen3.5-2B`.

Run cells top to bottom. Upload `quote-refuse-2026-10-16.train-x2f.jsonl` when asked.
Download the zips at the end and drop them in the repo.

In [ ]:
!pip install -q "unsloth==2026.10.5" "transformers==5.17.0" || pip install -q unsloth

In [ ]:
import importlib

for pkg in ["unsloth", "torch", "transformers", "trl", "peft"]:
    try:
        mod = importlib.import_module(pkg)
        print(pkg, getattr(mod, "__version__", "(no __version__)"))
    except Exception as exc:
        print(pkg, "import failed:", repr(exc))

In [ ]:
import gc
import hashlib
import os
import shutil

import torch

BASE = "unsloth/Qwen3.5-2B"
EXPORT = "nekohon_q35_2b"
DATA = "quote-refuse-2026-10-16.train-x2f.jsonl"
EXPECTED = 417  # x2d + 72 highlight + refusal oversample (the x2f set)
SEED = 3407
REFUSAL = "Not in this book."


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


if not os.path.exists(DATA):
    from google.colab import files
    files.upload()
assert os.path.exists(DATA), "upload " + DATA
print(DATA, "present:", os.path.exists(DATA))

In [ ]:
def train_one(model_name, export_name, data_file, expected, seed):
    gc.collect()
    torch.cuda.empty_cache()

    from datasets import load_dataset
    raw = load_dataset("json", data_files=data_file, split="train")
    print(len(raw), "cards loaded from", data_file)
    assert len(raw) == expected, (
        "row count mismatch: %s has %d cards, expected %d" % (data_file, len(raw), expected))
    print("data sha256:", sha256_file(data_file))
    print("refusals in file:", sum(1 for i in range(len(raw))
                                   if raw[i]["messages"][-1]["content"].strip() == REFUSAL),
          "of", len(raw))

    from unsloth import FastLanguageModel
    from unsloth.chat_templates import train_on_responses_only
    from trl import SFTConfig, SFTTrainer

    max_seq_length = 4096
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )

    # Qwen3.5 is a hybrid (Gated DeltaNet + Gated Attention); the linear
    # blocks may not expose q_proj/k_proj/... . Take the intersection of the
    # x2f target list with the modules that actually exist, and say which.
    wanted = ["q_proj", "k_proj", "v_proj", "o_proj",
              "gate_proj", "up_proj", "down_proj"]
    present = {n.split(".")[-1] for n, _ in model.named_modules()}
    targets = [t for t in wanted if t in present]
    if not targets:
        targets = "all-linear"
        print("no standard projection names found; falling back to all-linear")
    print("lora targets:", targets)

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        target_modules=targets,
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )

    def render(messages, gen=False):
        """Qwen3.5 small models default to no thinking; force it off anyway."""
        last = None
        for extra in ({"chat_template_kwargs": {"enable_thinking": False}}, {}):
            try:
                return tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=gen, **extra)
            except Exception as exc:
                last = exc
        raise last

    print("chat_template head:", (tokenizer.chat_template or "")[:200].replace("\n", " "))
    sample = render(raw[0]["messages"])
    assert "<|im_start|>assistant" in sample, (
        "assistant marker missing from the rendered template; paste this cell's output")
    print("render head:", sample[:240].replace("\n", " "))

    dataset = raw.map(lambda row: {"text": render(row["messages"])})

    lens = sorted(len(tokenizer(d["text"])["input_ids"]) for d in dataset)
    print("token lens p50/p90/max:", lens[len(lens) // 2],
          lens[int(.9 * len(lens)) - 1], lens[-1])
    assert lens[-1] <= max_seq_length, "truncation would cut the tail (EOS)"
    epochs = max(3, round(1968 / len(raw)))
    print("epochs:", epochs, "for", len(raw), "rows (~", epochs * len(raw), "presentations)")

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=epochs,
            max_length=max_seq_length,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=seed,
            output_dir="outputs",
            save_strategy="epoch",
            save_total_limit=4,
            report_to="none",
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("rows after responses-only prep:", len(trainer.train_dataset), "of", expected)
    assert len(trainer.train_dataset) == expected, "truncation canary dropped rows"

    batch = trainer.data_collator([trainer.train_dataset[0]])
    keep = [t.item() for t, l in zip(batch["input_ids"][0], batch["labels"][0]) if l != -100]
    sup = tokenizer.decode(keep)
    print("supervised tokens:", len(keep), "of", batch["labels"].shape[1])
    print("supervised tail:", repr(sup[-120:]))
    assert "<|im_end|>" in sup, "end token not supervised - do not train"

    print("config:", trainer.args.max_length, trainer.args.max_steps,
          trainer.args.seed, trainer.args.lr_scheduler_type, trainer.args.learning_rate)
    trainer.train()

    # STOP-CHECK before exporting: one quote + one gloss, then silence.
    model.eval()
    probe = render(raw[0]["messages"][:2], gen=True)
    inputs = tokenizer(probe, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    print("STOP-CHECK reply:", reply[:600])
    print("STOP-CHECK:", "SANE" if "<|im_end|>" in reply else "LONG OUTPUT — tell the agent")

    model.save_pretrained(export_name + "_lora")
    tokenizer.save_pretrained(export_name + "_lora")

    from google.colab import files

    for quant in ("q4_k_m", "q8_0"):
        out_dir = export_name + "_" + quant
        try:
            model.save_pretrained_gguf(out_dir, tokenizer, quantization_method=quant)
            shutil.make_archive(out_dir, "zip", out_dir)
            files.download(out_dir + ".zip")
        except Exception as exc:
            print(quant, "export failed:", repr(exc))

    try:
        merged = export_name + "_merged"
        model.save_pretrained_merged(merged, tokenizer, save_method="merged_16bit")
        shutil.make_archive(merged, "zip", merged)
        files.download(merged + ".zip")
    except Exception as exc:
        print("merged export failed:", repr(exc))

    print("done:", export_name)

In [ ]:
# One run, same shape as x2f.
train_one(BASE, EXPORT, DATA, EXPECTED, SEED)

## Send back

1. `nekohon_q35_2b_q4_k_m.zip` (the one the engine would use)
2. `nekohon_q35_2b_merged.zip` (fallback if Ollama refuses the GGUF)
3. The `STOP-CHECK` line and the final loss line

Known risk: Unsloth's docs say Qwen3.5 GGUFs need a separate `mmproj` file
and "currently no Qwen3.5 GGUF works in Ollama". The merged 16-bit zip is
the fallback for that; do not delete it.